In [15]:
%pip install statsmodels pmdarima -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel to use updated packages.


In [16]:
df = pd.read_csv('../data/processed/df_clustered.csv')
year_cols = list(range(2001, 2025))

print(f'Shape : {df.shape}')
print(f'\nDistribusi kluster:')
print(df.groupby(['cluster','cluster_name']).size().reset_index(name='jumlah'))

Shape : (514, 58)

Distribusi kluster:
   cluster cluster_name  jumlah
0        0         High      10
1        1         Zero     369
2        2    Declining      21
3        3     Moderate     114


In [17]:
cluster_series = {}

# Convert year_cols to strings to match DataFrame column names
string_year_cols = [str(year) for year in year_cols]

for c in sorted(df['cluster'].unique()):
    nama   = df[df['cluster'] == c]['cluster_name'].iloc[0]
    # Use string_year_cols for column selection
    total  = df[df['cluster'] == c][string_year_cols].sum(axis=0)
    # The index of 'total' should still be the original integer years for consistency in later plots/analysis
    total.index = year_cols
    cluster_series[c] = {'nama': nama, 'series': total}
    print(f'Kluster {c} — {nama}: total {total.sum():,.0f} ha')

Kluster 0 — High: total 647,235 ha
Kluster 1 — Zero: total 0 ha
Kluster 2 — Declining: total 870,106 ha
Kluster 3 — Moderate: total 790,712 ha


In [18]:
colors = ['#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig = go.Figure()

for c, data in cluster_series.items():
    fig.add_trace(go.Scatter(
        x=year_cols,
        y=data['series'].values,
        mode='lines+markers',
        name=f"Kluster {c} — {data['nama']}",
        line=dict(color=colors[c % len(colors)], width=2)
    ))

fig.add_vline(x=2011, line_dash='dash', line_color='gray',
              annotation_text='Moratorium 2011')
fig.update_layout(
    title='Total Deforestasi per Kluster (2001–2024)',
    xaxis_title='Tahun',
    yaxis_title='Deforestasi (ha)',
    template='plotly_white',
    height=450
)
fig.show()

In [19]:
print("=== UJI STASIONERITAS (ADF TEST) ===\n")

for c, data in cluster_series.items():

    series = data['series']

    if series.sum() == 0:
        print(f"Cluster {c} ({data['nama']}) : Tidak diuji (semua nol)")
        continue

    result = adfuller(series)

    print(f"Cluster {c} ({data['nama']})")
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"p-value       : {result[1]:.4f}")

    if result[1] < 0.05:
        print("Status        : Stasioner\n")
    else:
        print("Status        : Tidak stasioner\n")

=== UJI STASIONERITAS (ADF TEST) ===

Cluster 0 (High)
ADF Statistic : -2.6471
p-value       : 0.0837
Status        : Tidak stasioner

Cluster 1 (Zero) : Tidak diuji (semua nol)
Cluster 2 (Declining)
ADF Statistic : -0.9039
p-value       : 0.7867
Status        : Tidak stasioner

Cluster 3 (Moderate)
ADF Statistic : -2.7551
p-value       : 0.0650
Status        : Tidak stasioner



In [30]:
train_years = list(range(2001, 2020))
test_years  = list(range(2020, 2025))

print('Train: 2001–2019 (19 tahun)')
print('Test : 2020–2024 (5 tahun)')

for c, data in cluster_series.items():

    series = data['series']

    train_vals = np.array([series[y] for y in train_years], dtype=float)
    test_vals  = np.array([series[y] for y in test_years], dtype=float)

    cluster_series[c]['train'] = train_vals
    cluster_series[c]['test']  = test_vals

    print(f"\nKluster {c} — {data['nama']}")
    print(f"  Train sum : {train_vals.sum():,.0f} ha")
    print(f"  Test sum  : {test_vals.sum():,.0f} ha")

    if train_vals.sum() == 0:
        print("  -> Cluster Zero, SARIMA akan dilewati.")

Train: 2001–2019 (19 tahun)
Test : 2020–2024 (5 tahun)

Kluster 0 — High
  Train sum : 583,629 ha
  Test sum  : 63,606 ha

Kluster 1 — Zero
  Train sum : 0 ha
  Test sum  : 0 ha
  -> Cluster Zero, SARIMA akan dilewati.

Kluster 2 — Declining
  Train sum : 843,454 ha
  Test sum  : 26,652 ha

Kluster 3 — Moderate
  Train sum : 745,319 ha
  Test sum  : 45,393 ha


In [41]:
forecast_results = {}

for c, data in cluster_series.items():

    nama = data['nama']
    train_vals = np.array(data['train'], dtype=float)
    test_vals  = np.array(data['test'], dtype=float)

    # ===========================
    # Kluster Zero tidak diforecast
    # ===========================
    if train_vals.sum() == 0:
        print(f'Kluster {c} — {nama}: dilewati (semua nol)\n')

        forecast_results[c] = {
            'nama': nama,
            'rmse': 0,
            'mae': 0,
            'aic': np.nan,
            'order': None,
            'seasonal_order': None,
            'model': None,
            'pred_test': np.zeros(len(test_vals))
        }
        continue

    try:

        candidates = []

        orders = [
            (p,d,q)
            for p in range(4)
            for d in range(2)
            for q in range(4)
        ]

        seasonal_orders = [(0,0,0,0)] + [
            (P,D,Q,4)
            for P in range(3)
            for D in range(2)
            for Q in range(3)
        ]

        for order in orders:

            for seasonal_order in seasonal_orders:

                try:

                    model = SARIMAX(
                        train_vals,
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    )

                    fitted = model.fit(disp=False)

                    pred = fitted.forecast(steps=len(test_vals))
                    pred = np.clip(pred, 0, None)

                    rmse = np.sqrt(mean_squared_error(test_vals, pred))
                    mae  = mean_absolute_error(test_vals, pred)
                    aic  = fitted.aic

                    candidates.append((
                        rmse,
                        mae,
                        aic,
                        order,
                        seasonal_order,
                        fitted,
                        pred
                    ))

                except Exception:
                    continue

        if len(candidates) == 0:
            raise ValueError("Tidak ada model SARIMA yang valid.")

        # Prioritas:
        # 1. RMSE terkecil
        # 2. MAE terkecil
        # 3. AIC terkecil
        candidates.sort(key=lambda x: (x[0], x[1], x[2]))

        (
            best_rmse,
            best_mae,
            best_aic,
            best_order,
            best_seasonal,
            best_model,
            best_pred
        ) = candidates[0]

        forecast_results[c] = {
            'nama': nama,
            'model': best_model,
            'pred_test': best_pred,
            'rmse': best_rmse,
            'mae': best_mae,
            'aic': best_aic,
            'order': best_order,
            'seasonal_order': best_seasonal
        }

        print(f'Kluster {c} — {nama}')
        print(f'  Model terbaik : order={best_order}, seasonal_order={best_seasonal}')
        print(f'  RMSE          : {best_rmse:,.2f} ha')
        print(f'  MAE           : {best_mae:,.2f} ha')
        print(f'  AIC           : {best_aic:.2f}\n')

    except Exception as e:

        print(f'Kluster {c} — {nama}: ERROR — {e}\n')

Kluster 0 — High
  Model terbaik : order=(0, 0, 3), seasonal_order=(0, 1, 0, 4)
  RMSE          : 4,123.53 ha
  MAE           : 3,390.65 ha
  AIC           : 265.21

Kluster 1 — Zero: dilewati (semua nol)

Kluster 2 — Declining
  Model terbaik : order=(2, 0, 2), seasonal_order=(1, 0, 0, 4)
  RMSE          : 1,123.65 ha
  MAE           : 826.12 ha
  AIC           : 276.24

Kluster 3 — Moderate
  Model terbaik : order=(1, 1, 0), seasonal_order=(1, 0, 1, 4)
  RMSE          : 2,233.64 ha
  MAE           : 1,707.43 ha
  AIC           : 297.97



In [53]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        f"Kluster {c} — {d['nama']}" +
        (" (Tanpa Forecast)" if d['nama']=="Zero" else "")
        for c, d in cluster_series.items()
    ]
)

positions = [(1,1),(1,2),(2,1),(2,2)]

for idx, (c, data) in enumerate(cluster_series.items()):

    row, col = positions[idx]

    aktual = data['test']
    pred   = forecast_results[c]['pred_test']

    # Data aktual
    fig.add_trace(
        go.Scatter(
            x=test_years,
            y=aktual,
            mode='lines+markers',
            name='Aktual',
            line=dict(color=colors[c % len(colors)], width=3),
            marker=dict(size=8),
            showlegend=(idx==0)
        ),
        row=row,
        col=col
    )

    # Prediksi
    fig.add_trace(
        go.Scatter(
            x=test_years,
            y=pred,
            mode='lines+markers',
            name='Prediksi',
            line=dict(color='black', dash='dash', width=3),
            marker=dict(size=8),
            showlegend=(idx==0)
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title='Perbandingan Aktual dan Prediksi SARIMA per Klaster (2020–2024)',
    template='plotly_white',
    height=700,
    width=1100,
    legend=dict(
        orientation='h',
        y=1.08,
        x=0.5,
        xanchor='center'
    )
)

fig.update_xaxes(
    title_text='Tahun',
    tickmode='linear',
    dtick=1
)

fig.update_yaxes(
    title_text='Deforestasi (ha)',
    rangemode='tozero'
)

fig.show()

In [75]:
future_years = list(range(2025, 2031))
for c, data in cluster_series.items():
    nama        = data['nama']
    hist_vals   = data['series'].values
    future_vals = forecast_results[c]['future_pred']
    color       = colors[c % len(colors)]

    # Historis
    fig.add_trace(go.Scatter(
        x=year_cols, y=hist_vals,
        mode='lines+markers',
        name=f'{nama} (Historis)',
        line=dict(color=color, width=2)
    ))

    # Forecast
    fig.add_trace(go.Scatter(
        x=future_years, y=future_vals,
        mode='lines+markers',
        name=f'{nama} (Forecast)',
        line=dict(color=color, dash='dash', width=2)
    ))

fig.add_vline(x=2024, line_dash='dot', line_color='gray',
              annotation_text='Batas Historis')
fig.update_layout(
    title='Proyeksi Deforestasi Sawit per Kluster 2025-2030',
    xaxis_title='Tahun',
    yaxis_title='Deforestasi (ha)',
    template='plotly_white',
    height=500
)
fig.show()

In [76]:
# Ringkasan hasil forecast tiap klaster
for c, res in forecast_results.items():

    print(f"\nKluster {c} — {res['nama']}")

    if 'future_pred' not in res:
        print("Tidak ada forecast")
        continue

    df = pd.DataFrame({
        'Tahun': future_years,
        'Forecast (ha)': np.round(res['future_pred'], 2)
    })

    display(df)

    print(f"Total 2025–2030 : {df['Forecast (ha)'].sum():,.2f} ha")
    print(f"Rata-rata/tahun : {df['Forecast (ha)'].mean():,.2f} ha")


Kluster 0 — High


,Tahun,Forecast (ha)
0,2025,12539.76
1,2026,13399.85
2,2027,17063.29
3,2028,15346.49
4,2029,12539.76
5,2030,13399.85


Total 2025–2030 : 84,289.00 ha
Rata-rata/tahun : 14,048.17 ha

Kluster 1 — Zero


,Tahun,Forecast (ha)
0,2025,0.0
1,2026,0.0
2,2027,0.0
3,2028,0.0
4,2029,0.0
5,2030,0.0


Total 2025–2030 : 0.00 ha
Rata-rata/tahun : 0.00 ha

Kluster 2 — Declining


,Tahun,Forecast (ha)
0,2025,6404.00
1,2026,7660.74
2,2027,7285.61
3,2028,6087.64
4,2029,7522.87
5,2030,8006.21


Total 2025–2030 : 42,967.07 ha
Rata-rata/tahun : 7,161.18 ha

Kluster 3 — Moderate


,Tahun,Forecast (ha)
0,2025,10609.48
1,2026,11875.44
2,2027,11325.59
3,2028,10681.12
4,2029,11512.57
5,2030,10747.38


Total 2025–2030 : 66,751.58 ha
Rata-rata/tahun : 11,125.26 ha


In [62]:
summary = pd.DataFrame([
    {
        "Cluster": c,
        "Nama": r["nama"],
        "Order": r.get("order", "-"),
        "Seasonal": r.get("seasonal_order", "-"),
        "RMSE": round(r["rmse"], 2),
        "MAE": round(r["mae"], 2),
        "AIC": r.get("aic", np.nan)
    }
    for c, r in forecast_results.items()
])

summary["AIC"] = summary["AIC"].fillna("-")

df_eval = pd.DataFrame(summary)
print('EVALUASI MODEL SARIMA')
df_eval

EVALUASI MODEL SARIMA


,Cluster,Nama,Order,Seasonal,RMSE,MAE,AIC
0,0,High,"(0, 0, 3)","(0, 1, 0, 4)",4123.53,3390.65,265.209433
1,1,Zero,None,None,0.00,0.00,-
2,2,Declining,"(2, 0, 2)","(1, 0, 0, 4)",1123.65,826.12,276.236919
3,3,Moderate,"(1, 1, 0)","(1, 0, 1, 4)",2233.64,1707.43,297.96907


In [70]:
df_eval = pd.DataFrame([
    {
        'cluster': c,
        'cluster_name': res['nama'],
        'order': res.get('order'),
        'seasonal_order': res.get('seasonal_order'),
        'rmse_ha': round(res.get('rmse', 0), 2),
        'mae_ha': round(res.get('mae', 0), 2),
        'aic': round(res['aic'], 2) if res.get('aic') is not None else "None"
    }
    for c, res in forecast_results.items()
])

df_eval.to_csv(
    '../data/processed/evaluasi_sarima.csv',
    index=False
)

forecast_rows = []

for c, res in forecast_results.items():

    future_pred = res.get('future_pred', np.zeros(len(future_years)))

    for year, value in zip(future_years, future_pred):

        forecast_rows.append({
            'cluster': c,
            'cluster_name': res['nama'],
            'year': year,
            'forecast_ha': round(float(value), 2)
        })

df_forecast = pd.DataFrame(forecast_rows)

df_forecast.to_csv(
    '../data/processed/forecast_2025_2030.csv',
    index=False
)

print("evaluasi_sarima.csv berhasil diperbarui")
print(df_eval)

print()

print("forecast_2025_2030.csv berhasil diperbarui")
print(df_forecast.head())

print(f"\nTotal forecast tersimpan: {len(df_forecast)} baris")

evaluasi_sarima.csv berhasil diperbarui
   cluster cluster_name      order seasonal_order  rmse_ha   mae_ha     aic
0        0         High  (0, 0, 3)   (0, 1, 0, 4)  4123.53  3390.65  265.21
1        1         Zero       None           None     0.00     0.00     NaN
2        2    Declining  (2, 0, 2)   (1, 0, 0, 4)  1123.65   826.12  276.24
3        3     Moderate  (1, 1, 0)   (1, 0, 1, 4)  2233.64  1707.43  297.97

forecast_2025_2030.csv berhasil diperbarui
   cluster cluster_name  year  forecast_ha
0        0         High  2025     12539.76
1        0         High  2026     13399.85
2        0         High  2027     17063.29
3        0         High  2028     15346.49
4        0         High  2029     12539.76

Total forecast tersimpan: 24 baris
